# A12 · Inference, Quantization & Serving on AWS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a12_inference.ipynb)

Generating text is a different workload from training: a compute-bound prefill, then a memory-bound decode loop. Learn the KV-cache equation, why decode speed ≈ bandwidth / model size, how quantisation works numerically, and how to serve models on AWS GPUs, Inferentia or Bedrock.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a12). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Compute KV-cache size $2\,L\,n_{kv}\,d_{head}\,T\,b$ bytes and the effect of GQA.
- Explain prefill (compute-bound) vs decode (memory-bound) and estimate tokens per second.
- Quantise weights to int8 by hand and reason about int4 trade-offs.
- Implement temperature and top-p sampling, and serve a model with vLLM, SageMaker or Bedrock.

## The math

### KV-cache size

$$M_{KV} = 2 \cdot L \cdot n_{kv} \cdot d_{head} \cdot T \cdot b \cdot \text{bytes}$$

Factor 2 for K and V. Llama-2-7B-like shape: $L = 32,\ n_{kv} = 32,\ d_{head} = 128$, fp16.

**Worked example**

1. $T = 4096,\ b = 1$: $2\cdot32\cdot32\cdot128\cdot4096\cdot2 = 2{,}147{,}483{,}648$ B (= 2 GiB for one sequence)
2. batch of 16: 32 GiB, more than the 14 GB of weights (KV cache, not weights, limits batch size)
3. GQA with $n_{kv} = 8$: 0.5 GiB per sequence (4× more concurrent users)

### Decode speed bound

$$\text{tokens/s per sequence} \lesssim \frac{\text{memory bandwidth}}{\text{bytes of weights}}$$

Each decode step reads every weight once. Batching shares that read across many sequences.

**Worked example**

1. 7B fp16 = 14 GB on an A10G (600 GB/s): $600/14 \approx 43$ tokens/s upper bound
2. int4 (3.5 GB): $\approx 170$ tokens/s upper bound (why quantisation speeds up decode)
3. prefill of 1,000 tokens: $2N\cdot1000 = 1.4\times10^{13}$ FLOPs (≈ 0.1–0.4 s on the same GPU)

### Symmetric int8 quantisation

$$s = \frac{\max_i |w_i|}{127}, \qquad q_i = \mathrm{round}\!\left(\frac{w_i}{s}\right), \qquad \hat w_i = s\,q_i$$

$w = (0.12,\ -0.50,\ 0.31,\ 0.02)$.

**Worked example**

1. $s = 0.50/127 = 0.003937$
2. $q = \mathrm{round}(30.5,\ -127,\ 78.7,\ 5.1) = (30,\ -127,\ 79,\ 5)$ (4 bytes instead of 16)
3. $\hat w = (0.1181,\ -0.5000,\ 0.3110,\ 0.0197)$ (max error $\le s/2 = 0.002$)
4. one outlier weight inflates $s$ for everyone (why int4 uses small groups (e.g., 128 weights per scale) and methods such as AWQ/GPTQ)

### Temperature and top-p

$$p_i = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}, \qquad \text{top-}p:\ \text{smallest set } S \text{ with } \textstyle\sum_{i\in S}p_i \ge p$$

Logits $z = (2.0,\ 1.0,\ 0.1)$.

**Worked example**

1. $T = 1$: $(0.659,\ 0.242,\ 0.099)$
2. $T = 0.5$: $(0.864,\ 0.117,\ 0.019)$ (sharper)
3. top-p 0.9 at $T = 1$: keep $\{0, 1\}$ (0.901), renormalise to $(0.731,\ 0.269)$

In [ ]:
%pip install -q transformers

## Lab: Sampling, int8, KV cache speed-up, and serving

Reproduce the worked numbers, measure the KV-cache speed-up, then serve an open model with an OpenAI-compatible API.

### Step 1

Temperature and top-p sampling.

In [ ]:
import torch, time
def probs(logits, temperature=1.0):
    return (logits / temperature).softmax(-1)
def top_p_filter(p, top_p=0.9):
    sorted_p, idx = p.sort(descending=True)
    keep = sorted_p.cumsum(-1) - sorted_p < top_p        # keep tokens until the mass reaches top_p
    out = torch.zeros_like(p)
    out[idx[keep]] = sorted_p[keep]
    return out / out.sum()
z = torch.tensor([2.0, 1.0, 0.1])
print(probs(z, 1.0), probs(z, 0.5))         # [0.659 0.242 0.099] [0.864 0.117 0.019]
print(top_p_filter(probs(z), 0.9))          # [0.731 0.269 0.000]
print(torch.multinomial(top_p_filter(probs(z), 0.9), 10, replacement=True))

### Step 2

Symmetric int8 quantisation, then per-row quantisation of a real weight matrix.

In [ ]:
w = torch.tensor([0.12, -0.5, 0.31, 0.02])
s = w.abs().max() / 127
q = torch.round(w / s).to(torch.int8)
print(q, q.float() * s)                     # [30, -127, 79, 5] -> [0.1181, -0.5, 0.3110, 0.0197]

W = torch.randn(4096, 4096) * 0.02
scales = W.abs().amax(dim=1, keepdim=True) / 127      # one scale per output row
Wq = torch.round(W / scales).to(torch.int8)
err = (Wq.float() * scales - W).abs().mean() / W.abs().mean()
print(f"bytes: {W.numel()*2/1e6:.0f} MB fp16 -> {W.numel()/1e6:.0f} MB int8, mean rel. error {err:.3%}")

### Step 3

KV cache in action with GPT-2 (`pip install transformers`).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
ids = tok("The cloud is", return_tensors="pt").input_ids.to(device)
for use_cache in [False, True]:
    t = time.perf_counter()
    out = model.generate(ids, max_new_tokens=300, do_sample=False, use_cache=use_cache,
                         pad_token_id=tok.eos_token_id)
    print(f"use_cache={use_cache}: {time.perf_counter() - t:.2f}s")
L, h, d = model.config.n_layer, model.config.n_head, model.config.n_embd // model.config.n_head
print("KV bytes for 1024 tokens fp16:", 2 * L * h * d * 1024 * 2)    # 37,748,736

### Step 4

Serve an open model on an AWS GPU instance (e.g., g5.xlarge with a Deep Learning AMI). vLLM exposes an OpenAI-compatible API.

> Run this in a terminal on a GPU instance (e.g., EC2 g5.xlarge), not in Colab.

```bash
pip install vllm
vllm serve Qwen/Qwen2.5-1.5B-Instruct --max-model-len 4096 --gpu-memory-utilization 0.9 &
# wait for "Application startup complete", then:
curl -s localhost:8000/v1/chat/completions -H 'Content-Type: application/json' -d '{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "messages": [{"role": "user", "content": "Terangkan KV cache dalam dua ayat."}],
  "max_tokens": 120, "temperature": 0.7}'
```

### Step 5

Managed alternative: call a Bedrock model with the Converse API (no servers), using the CLI from module 18 or boto3.

> ⚠️ This step runs against your AWS account and may incur charges. Skip it if you are only using Colab.

In [ ]:
import boto3
brt = boto3.client("bedrock-runtime", region_name="ap-southeast-1")
resp = brt.converse(
    modelId="<inference-profile-id>",              # from: aws bedrock list-inference-profiles
    messages=[{"role": "user", "content": [{"text": "Explain the KV cache in two sentences."}]}],
    inferenceConfig={"maxTokens": 150, "temperature": 0.5},
)
print(resp["output"]["message"]["content"][0]["text"])
print(resp["usage"])                                # input/output tokens: what you're billed for

### Clean up

Stop or terminate the GPU instance as soon as you're done: a g5.xlarge bills about US$1/hour. Delete any SageMaker endpoint (endpoints bill continuously).

## Check yourself

**1. KV cache for $L = 40,\ n_{kv} = 8,\ d_{head} = 128,\ T = 8192$, fp16, batch 1?**

A. 1.34 GB  
B. 0.67 GB  
C. 5.4 GB  
D. 168 MB

<details><summary>Answer</summary>

**A.** $2\cdot40\cdot8\cdot128\cdot8192\cdot2 = 1{,}342{,}177{,}280$ B ≈ 1.34 GB.

</details>

**2. Why is single-sequence decoding memory-bound?**

A. Attention is O(T²)  
B. Each step does ~2 FLOPs per weight but must read every weight from memory  
C. The tokenizer is slow  
D. Sampling is expensive

<details><summary>Answer</summary>

**B.** Arithmetic intensity is about 1 FLOP per byte at batch 1, far below the GPU's compute/bandwidth ratio. Batching raises it.

</details>

**3. int8 symmetric quantisation with max $|w| = 2.54$. Scale $s$?**

A. 0.01  
B. 0.02  
C. 2.54  
D. 127

<details><summary>Answer</summary>

**B.** $s = 2.54/127 = 0.02$.

</details>